# HistoCNN: Colorectal Histopathology Patch Classification
### Deep Learning with PyTorch on Google Colab (T4 GPU)

This notebook trains **HistoCNN**, a lightweight custom 4-block Convolutional Neural Network (~308k parameters), from scratch on the **NCT-CRC-HE** dataset (100k non-overlapping H&E patches) across 9 tissue classes:
- **ADI**: Adipose tissue
- **BACK**: Background / clear glass
- **DEB**: Debris
- **LYM**: Lymphocytes
- **MUC**: Mucus
- **MUS**: Smooth muscle
- **NORM**: Normal colon mucosa
- **STR**: Cancer-associated stroma
- **TUM**: Colorectal adenocarcinoma epithelium


In [ ]:
# Step 1: Install Dependencies
!pip install -q torch torchvision datasets pillow matplotlib seaborn

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")


In [ ]:
# Step 2: Define Data Pipeline and Augmentations
import os
import random
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
from datasets import load_dataset

CLASSES = ["ADI", "BACK", "DEB", "LYM", "MUC", "MUS", "NORM", "STR", "TUM"]

def make_transforms(train: bool = True):
    if train:
        return T.Compose([
            T.Resize((64, 64)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.5),
            T.ToTensor(),
            T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ])
    return T.Compose([
        T.Resize((64, 64)),
        T.ToTensor(),
        T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])

class HFSubsetDataset(Dataset):
    def __init__(self, items, transform=None):
        self.items = items
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        image = item["image"]
        label = item["label"]
        if image.mode != "RGB":
            image = image.convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label


In [ ]:
# Step 3: Stream and Build Balanced Dataset Subset
PER_CLASS = 1000
NUM_CLASSES = len(CLASSES)

print("Streaming NCT-CRC-HE dataset from Hugging Face...")
ds = load_dataset("1aurent/NCT-CRC-HE", split="NCT_CRC_HE_100K", streaming=True)

class_buckets = {c: [] for c in range(NUM_CLASSES)}
for item in ds:
    c = item["label"]
    if len(class_buckets[c]) < PER_CLASS:
        class_buckets[c].append({"image": item["image"], "label": c})
    if all(len(v) >= PER_CLASS for v in class_buckets.values()):
        break

all_items = []
for c in range(NUM_CLASSES):
    all_items.extend(class_buckets[c])

random.seed(42)
random.shuffle(all_items)

split = int(len(all_items) * 0.85)
train_dataset = HFSubsetDataset(all_items[:split], make_transforms(True))
val_dataset = HFSubsetDataset(all_items[split:], make_transforms(False))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Train samples: {len(train_dataset):,} | Validation samples: {len(val_dataset):,}")


In [ ]:
# Step 4: Model Architecture (HistoCNN)
import torch.nn as nn

class HistoCNN(nn.Module):
    def __init__(self, num_classes: int = 9):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((2, 2)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 2 * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = HistoCNN(9).to(device)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"HistoCNN successfully instantiated with {trainable_params:,} trainable parameters.")


In [ ]:
# Step 5: Training Loop with Best Model Checkpointing
import copy

EPOCHS = 12
torch.manual_seed(42)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

best_acc = -1.0
best_loss = float("inf")
best_state = copy.deepcopy(model.state_dict())

for epoch in range(EPOCHS):
    model.train()
    t_loss, t_corr, t_total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        bs = labels.size(0)
        t_loss += loss.item() * bs
        t_corr += (logits.argmax(dim=1) == labels).sum().item()
        t_total += bs

    model.eval()
    v_loss, v_corr, v_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            bs = labels.size(0)
            v_loss += loss.item() * bs
            v_corr += (logits.argmax(dim=1) == labels).sum().item()
            v_total += bs

    train_acc = t_corr / t_total
    val_acc = v_corr / v_total
    val_loss = v_loss / v_total

    if val_acc > best_acc or (abs(val_acc - best_acc) < 1e-12 and val_loss < best_loss):
        best_acc = val_acc
        best_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())

    print(f"Epoch {epoch+1:02d}/{EPOCHS:02d} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% (Loss: {val_loss:.4f})")

model.load_state_dict(best_state)
print(f"
Restored Best Validation Accuracy: {best_acc*100:.2f}%")


In [ ]:
# Step 6: Detailed Evaluation Report
def evaluate_model(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            logits = model(images)
            all_preds.append(logits.argmax(dim=1).cpu())
            all_labels.append(labels)
    y_pred = torch.cat(all_preds)
    y_true = torch.cat(all_labels)

    acc = (y_true == y_pred).sum().item() / len(y_true)
    conf = torch.zeros(len(CLASSES), len(CLASSES), dtype=torch.long)
    for t, p in zip(y_true, y_pred):
        conf[t.item(), p.item()] += 1

    recalls = []
    for c in range(len(CLASSES)):
        cnt = (y_true == c).sum().item()
        if cnt > 0:
            recalls.append(((y_true == c) & (y_pred == c)).sum().item() / cnt)
    macro_rec = sum(recalls) / len(recalls) if recalls else 0.0
    return acc, macro_rec, conf

acc, macro_rec, conf = evaluate_model(model, val_loader, device)
print(f"Validation Accuracy: {acc*100:.2f}%")
print(f"Macro Recall       : {macro_rec*100:.2f}%")


In [ ]:
# Step 7: Plot Publication-Grade Confusion Matrix Heatmap
import matplotlib.pyplot as plt
import seaborn as sns

cm_np = conf.numpy()
cm_norm = cm_np.astype("float") / cm_np.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(9, 7.5), dpi=300)
sns.heatmap(
    cm_norm,
    annot=cm_np,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASSES,
    yticklabels=CLASSES,
    cbar_kws={"label": "Row-Normalized Recall"},
    linewidths=0.5,
    ax=ax,
)
ax.set_xlabel("Predicted Tissue Class", fontsize=11, fontweight="bold", labelpad=10)
ax.set_ylabel("True Tissue Class", fontsize=11, fontweight="bold", labelpad=10)
ax.set_title(f"HistoCNN Confusion Matrix
Accuracy: {acc*100:.2f}% | Macro Recall: {macro_rec*100:.2f}%", fontsize=13, fontweight="bold", pad=14)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=300)
plt.show()


In [ ]:
# Step 8: Save Model Weights
import os
os.makedirs("weights", exist_ok=True)
weights_path = "weights/histocnn_best.pth"
torch.save(model.state_dict(), weights_path)
print(f"Model weights saved to {weights_path} ({os.path.getsize(weights_path)/(1024*1024):.2f} MB)")

# Optional: Download checkpoint to your local machine from Colab
try:
    from google.colab import files
    files.download(weights_path)
except ImportError:
    pass
